In [28]:
# ======================== CELL 1: IMPORTS ========================
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
DATA_PATH = '/kaggle/input/competitions/datathon-2026-round-1'
print('Imports done ✅')

Imports done ✅


In [29]:
# ======================== CELL 2: LOAD DATA ========================
def load(name, **kwargs):
    return pd.read_csv(f'{DATA_PATH}/{name}', **kwargs)

sales       = load('sales.csv',             parse_dates=['Date'])
sub         = load('sample_submission.csv', parse_dates=['Date'])
traffic     = load('web_traffic.csv',       parse_dates=['date'])
orders      = load('orders.csv',            parse_dates=['order_date'])
order_items = load('order_items.csv')
payments    = load('payments.csv')
returns_df  = load('returns.csv',           parse_dates=['return_date'])
reviews_df  = load('reviews.csv',           parse_dates=['review_date'])
promotions  = load('promotions.csv',        parse_dates=['start_date','end_date'])
inventory   = load('inventory.csv',         parse_dates=['snapshot_date'])

sales  = sales.sort_values('Date').reset_index(drop=True)
rev_s  = sales.set_index('Date')['Revenue']
cogs_s = sales.set_index('Date')['COGS']

print(f'Sales: {len(sales)} rows | {sales["Date"].min().date()} → {sales["Date"].max().date()}')
print(f'Test:  {sub["Date"].min().date()} → {sub["Date"].max().date()} ({len(sub)} days)')
print(f'Revenue mean={sales["Revenue"].mean():,.0f} | std={sales["Revenue"].std():,.0f}')
print('\nRevenue mean theo năm:')
for yr in range(2013, 2023):
    print(f'  {yr}: {sales[sales["Date"].dt.year==yr]["Revenue"].mean():>12,.0f}')

Sales: 3833 rows | 2012-07-04 → 2022-12-31
Test:  2023-01-01 → 2024-07-01 (548 days)
Revenue mean=4,286,584 | std=2,624,840

Revenue mean theo năm:
  2013:    4,540,190
  2014:    5,128,345
  2015:    5,177,901
  2016:    5,750,384
  2017:    5,236,067
  2018:    5,068,829
  2019:    3,114,524
  2020:    2,881,181
  2021:    2,857,643
  2022:    3,204,791


In [30]:
# ======================== CELL 3 REPLACEMENT (V8 FIX) ========================
# Vấn đề V8: yoy_2024=1.0433 → 2024 mean=4.79M (vô lý cao hơn train mean)
# Fix: grid search yoy_2023 trên val 2022, hard-cap yoy_2024 = median(recent yoys)

def calc_yoy(series, y1, y2):
    m1 = series[series.index.year == y1].mean()
    m2 = series[series.index.year == y2].mean()
    return float(m2 / m1) if m1 > 0 else 1.0

print('YoY Revenue thực tế:')
yoy_history = {}
for y in range(2014, 2023):
    yy = calc_yoy(rev_s, y-1, y)
    yoy_history[y] = yy
    print(f'  {y-1}→{y}: {yy:.4f}x')

# ── Grid search yoy_2023 trên val 2022 ──────────────────────────────────────
print('\n🔎 Grid search yoy_2023 trên val 2022...')
val22_data = sales[sales['Date'].dt.year == 2022]
best_sn_mae, best_yoy_23 = float('inf'), 1.0

for yoy_try in np.arange(0.85, 1.16, 0.02):
    preds = []
    for d in val22_data['Date']:
        rv = [rev_s.get(d - pd.Timedelta(days=l), np.nan) for l in [364,365,366]]
        rv = [v for v in rv if not np.isnan(v)]
        base = np.mean(rv) if rv else rev_s[rev_s.index.year==2021].mean()
        preds.append(base * yoy_try)
    mae_try = mean_absolute_error(val22_data['Revenue'], preds)
    if mae_try < best_sn_mae:
        best_sn_mae, best_yoy_23 = mae_try, yoy_try
    print(f'  yoy={yoy_try:.2f} → SN val22 MAE={mae_try:>10,.0f}')

# yoy_2023 = best found on val 2022
yoy_2023_rev = best_yoy_23
yoy_2023_cogs = float(np.clip(calc_yoy(cogs_s, 2021, 2022), 0.90, 1.15)) * (best_yoy_23 / calc_yoy(rev_s, 2021, 2022))

# ── yoy_2024: dùng median của recent yoys, KHÔNG dùng weighted avg ──────────
# Median robust với recovery year outlier (2021→2022: 1.12x)
recent_yoys_r = [yoy_history[y] for y in [2019,2020,2021,2022]]
recent_yoys_c = [calc_yoy(cogs_s, y-1, y) for y in [2019,2020,2021,2022]]
yoy_2024_rev  = float(np.clip(np.median(recent_yoys_r), 0.88, 1.02))
yoy_2024_cogs = float(np.clip(np.median(recent_yoys_c), 0.88, 1.02))

# Backward compat
yoy_rev  = yoy_2023_rev
yoy_cogs = yoy_2023_cogs

print(f'\n✅ yoy_2023: Rev={yoy_2023_rev:.4f}x | COGS={yoy_2023_cogs:.4f}x')
print(f'   yoy_2024: Rev={yoy_2024_rev:.4f}x | COGS={yoy_2024_cogs:.4f}x')
print(f'   (yoy_2024 = median of 2018→2022 yoys, not weighted avg)')

# ── SN Robust Buffer ─────────────────────────────────────────────────────────
sn_r = rev_s.copy()
sn_c = cogs_s.copy()
test_dates      = sorted(pd.to_datetime(sub['Date'].tolist()))
sn_rev_preds    = []
sn_cogs_preds   = []

for date in test_dates:
    r_1y = [sn_r.get(date - pd.Timedelta(days=l), np.nan) for l in [364,365,366]]
    r_2y = [sn_r.get(date - pd.Timedelta(days=l), np.nan) for l in [728,730,732]]
    c_1y = [sn_c.get(date - pd.Timedelta(days=l), np.nan) for l in [364,365,366]]
    c_2y = [sn_c.get(date - pd.Timedelta(days=l), np.nan) for l in [728,730,732]]
    r_1y = [v for v in r_1y if not np.isnan(v)]
    r_2y = [v for v in r_2y if not np.isnan(v)]
    c_1y = [v for v in c_1y if not np.isnan(v)]
    c_2y = [v for v in c_2y if not np.isnan(v)]

    yoy_r = yoy_2024_rev  if date.year >= 2024 else yoy_2023_rev
    yoy_c = yoy_2024_cogs if date.year >= 2024 else yoy_2023_cogs

    # 1yr base (2023 → dùng 2022 thật | 2024 → dùng SN 2023)
    base_r = np.mean(r_1y) if r_1y else sn_r.tail(90).mean()
    base_c = np.mean(c_1y) if c_1y else sn_c.tail(90).mean()

    # 2yr cross-check (sanity)
    cross_r = np.mean(r_2y) * (yoy_r ** 2) if r_2y else None

    # Final: nếu 1yr base và 2yr cross-check quá khác nhau → dùng trung bình
    pr = base_r * yoy_r
    if cross_r is not None and abs(pr - cross_r) / (pr + 1e-5) > 0.30:
        pr = 0.60 * pr + 0.40 * cross_r  # blend khi diverge >30%

    pc = base_c * yoy_c
    pr = max(0, pr)
    pc = max(0, min(pc, pr * 0.97))

    sn_rev_preds.append(pr)
    sn_cogs_preds.append(pc)
    sn_r[date] = pr
    sn_c[date] = pc

for yr in [2023, 2024]:
    m = [v for v, d in zip(sn_rev_preds, test_dates) if d.year == yr]
    if m: print(f'SN Rev {yr}: {np.mean(m):,.0f}')
print(f'SN Rev all:  {np.mean(sn_rev_preds):,.0f}')
print(f'(Target: similar to 2022 mean = 3,204,791)')

YoY Revenue thực tế:
  2013→2014: 1.1295x
  2014→2015: 1.0097x
  2015→2016: 1.1106x
  2016→2017: 0.9106x
  2017→2018: 0.9681x
  2018→2019: 0.6144x
  2019→2020: 0.9251x
  2020→2021: 0.9918x
  2021→2022: 1.1215x

🔎 Grid search yoy_2023 trên val 2022...
  yoy=0.85 → SN val22 MAE=   946,595
  yoy=0.87 → SN val22 MAE=   911,850
  yoy=0.89 → SN val22 MAE=   878,947
  yoy=0.91 → SN val22 MAE=   849,002
  yoy=0.93 → SN val22 MAE=   822,142
  yoy=0.95 → SN val22 MAE=   797,391
  yoy=0.97 → SN val22 MAE=   776,762
  yoy=0.99 → SN val22 MAE=   760,247
  yoy=1.01 → SN val22 MAE=   746,049
  yoy=1.03 → SN val22 MAE=   734,431
  yoy=1.05 → SN val22 MAE=   729,148
  yoy=1.07 → SN val22 MAE=   727,270
  yoy=1.09 → SN val22 MAE=   727,537
  yoy=1.11 → SN val22 MAE=   729,193
  yoy=1.13 → SN val22 MAE=   736,202
  yoy=1.15 → SN val22 MAE=   748,478

✅ yoy_2023: Rev=1.0700x | COGS=1.0345x
   yoy_2024: Rev=0.9585x | COGS=0.9721x
   (yoy_2024 = median of 2018→2022 yoys, not weighted avg)
SN Rev 2023: 3,390

In [31]:
# ======================== CELL 4: AUXILIARY TABLES ========================
print('Building auxiliary tables...')

# ── Promotion calendar ──
all_dates_range = pd.date_range(sales['Date'].min(), '2024-07-01', freq='D')
promo_records = []
for d in all_dates_range:
    active = promotions[(promotions['start_date'] <= d) & (promotions['end_date'] >= d)]
    n = len(active)
    # Days to next promo (demand pull-forward signal)
    future = promotions[promotions['start_date'] > d]
    days_to_next = int(min((future['start_date'].min() - d).days, 60)) if len(future) > 0 else 60
    promo_records.append({
        'Date':          d,
        'n_promos':      n,
        'has_promo':     int(n > 0),
        'max_discount':  float(active['discount_value'].max())  if n > 0 else 0.0,
        'avg_discount':  float(active['discount_value'].mean()) if n > 0 else 0.0,
        'n_stackable':   int(active['stackable_flag'].sum())    if n > 0 else 0,
        'has_pct_promo': int((active['promo_type'] == 'percentage').any()) if n > 0 else 0,
        'has_multi_promo': int(n >= 2),
        'days_to_next_promo': days_to_next,
    })
df_promo = pd.DataFrame(promo_records).set_index('Date')
PROMO_COLS = list(df_promo.columns)
print(f'  Promo calendar: {df_promo.shape}, promo days: {df_promo["has_promo"].sum()}')

# ── Inventory (monthly → daily lookup) ──
monthly_inv = inventory.groupby('snapshot_date').agg(
    inv_fill_rate    = ('fill_rate',         'mean'),
    inv_stockout_pct = ('stockout_flag',     'mean'),
    inv_days_supply  = ('days_of_supply',    'mean'),
    inv_sell_thru    = ('sell_through_rate', 'mean'),
    inv_reorder_pct  = ('reorder_flag',      'mean'),
).reset_index()
monthly_inv['Month'] = monthly_inv['snapshot_date'].dt.to_period('M')
monthly_inv = monthly_inv.set_index('Month').drop(columns=['snapshot_date'])
INV_COLS = list(monthly_inv.columns)
INV_DEF  = {'inv_fill_rate':0.90,'inv_stockout_pct':0.10,'inv_days_supply':30.0,
             'inv_sell_thru':0.50,'inv_reorder_pct':0.15}

# ── Traffic (daily) ──
traf_agg_cfg = {'sessions':'sum','unique_visitors':'sum','page_views':'sum',
                'bounce_rate':'mean','avg_session_duration_sec':'mean'}
traf_agg_cfg = {k:v for k,v in traf_agg_cfg.items() if k in traffic.columns}
traf_day = traffic.groupby('date').agg(**{k:(k,v) for k,v in traf_agg_cfg.items()})\
                  .reset_index().rename(columns={'date':'Date'}).set_index('Date')
if 'page_views' in traf_day and 'sessions' in traf_day:
    traf_day['pages_per_session'] = traf_day['page_views'] / (traf_day['sessions'] + 1e-5)
if 'bounce_rate' in traf_day and 'pages_per_session' in traf_day:
    traf_day['engagement'] = traf_day['pages_per_session'] * (1 - traf_day['bounce_rate'])
# Source breakdown
if 'traffic_source' in traffic.columns:
    src = traffic.groupby(['date','traffic_source'])['sessions'].sum().unstack(fill_value=0).reset_index()
    src.columns = ['Date'] + [f'src_{c}' for c in src.columns[1:]]
    total = src[[c for c in src.columns if c.startswith('src_')]].sum(axis=1)
    for c in [col for col in src.columns if col.startswith('src_')]:
        src[c+'_pct'] = src[c] / (total + 1e-5)
    src = src.set_index('Date')
    traf_day = traf_day.join(src, how='left')
TRAF_COLS = list(traf_day.columns)
print(f'  Traffic cols: {len(TRAF_COLS)} | {TRAF_COLS[:5]}...')

# ── Order signals (daily) ──
oi = order_items.merge(orders[['order_id','order_date','customer_id','order_status']],
                       on='order_id', how='left')
oi['has_promo']    = (~oi['promo_id'].isna()).astype(int)
oi['discount_pct'] = oi['discount_amount'] / (oi['quantity'] * oi['unit_price'] + 1e-5)
daily_oi = oi.groupby('order_date').agg(
    n_orders_raw   = ('order_id',     'nunique'),
    promo_item_pct = ('has_promo',    'mean'),
    avg_disc_pct   = ('discount_pct', 'mean'),
    avg_unit_price = ('unit_price',   'mean'),
    total_qty      = ('quantity',     'sum'),
).reset_index().rename(columns={'order_date':'Date'})
daily_basket = (
    payments.merge(orders[['order_id','order_date']], on='order_id')
    .groupby('order_date').agg(avg_basket=('payment_value','mean'),
                                median_basket=('payment_value','median'))
    .reset_index().rename(columns={'order_date':'Date'})
)
daily_cancel = orders.groupby('order_date').agg(
    cancel_rate=('order_status', lambda x: (x=='cancelled').mean())
).reset_index().rename(columns={'order_date':'Date'})
first_orders = orders.groupby('customer_id')['order_date'].min().reset_index()
first_orders.columns = ['customer_id','first_date']
ord_ext = orders.merge(first_orders, on='customer_id')
ord_ext['is_new'] = (ord_ext['order_date'] == ord_ext['first_date']).astype(int)
daily_newcust = ord_ext.groupby('order_date').agg(
    new_cust_pct=('is_new','mean')
).reset_index().rename(columns={'order_date':'Date'})
daily_oi = daily_oi.merge(daily_basket,  on='Date', how='left')\
                   .merge(daily_cancel,  on='Date', how='left')\
                   .merge(daily_newcust, on='Date', how='left')\
                   .set_index('Date')
ORDER_COLS = list(daily_oi.columns)

# ── Returns & Reviews ──
daily_ret = returns_df.groupby('return_date').agg(
    n_returns=('return_id','count'), refund_total=('refund_amount','sum')
).reset_index().rename(columns={'return_date':'Date'}).set_index('Date')

daily_rev_sig = reviews_df.groupby('review_date').agg(
    n_reviews=('review_id','count'), avg_rating=('rating','mean'),
    pct_5star=('rating', lambda x: (x==5).mean()),
    pct_1star=('rating', lambda x: (x==1).mean()),
).reset_index().rename(columns={'review_date':'Date'}).set_index('Date')

print('All auxiliary tables done ✅')

Building auxiliary tables...
  Promo calendar: (4381, 8), promo days: 1707
  Traffic cols: 19 | ['sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec']...
All auxiliary tables done ✅


In [32]:
# ======================== CELL 5: HOLIDAY FEATURES ========================
TET_DATES = set(pd.to_datetime([
    '2013-02-10','2014-01-31','2015-02-19','2016-02-08',
    '2017-01-28','2018-02-16','2019-02-05','2020-01-25',
    '2021-02-12','2022-02-01','2023-01-22','2024-02-10',
]))
ALL_TET  = sorted(TET_DATES)
PRE_TET  = set(d for t in ALL_TET for d in pd.date_range(end=t-pd.Timedelta(1), periods=45))
POST_TET = set(d for t in ALL_TET for d in pd.date_range(start=t+pd.Timedelta(1), periods=15))

def holiday_features(d):
    d = pd.Timestamp(d)
    f = {
        'is_tet':        int(d in TET_DATES),
        'is_pre_tet':    int(d in PRE_TET),
        'is_post_tet':   int(d in POST_TET),
        'is_11_11':      int(d.month==11 and d.day==11),
        'is_12_12':      int(d.month==12 and d.day==12),
        'is_10_10':      int(d.month==10 and d.day==10),
        'is_9_9':        int(d.month==9  and d.day==9),
        'is_6_6':        int(d.month==6  and d.day==6),
        'is_xmas':       int(d.month==12 and 23<=d.day<=26),
        'is_new_year':   int((d.month==12 and d.day==31) or (d.month==1 and d.day<=3)),
        'is_womens_day': int(d.month==3 and d.day==8),
        'is_nat_day':    int(d.month==9 and d.day==2),
        'is_black_fri':  int(d.month==11 and d.dayofweek==4 and d.day>=25),
        'is_payday':     int(d.day>=28 or d.day<=5),
        'is_mid_month':  int(13<=d.day<=17),
        'is_pre_1111':   int(d.month==11 and 1<=d.day<=10),
        'is_pre_1212':   int(d.month==12 and 1<=d.day<=11),
        'is_post_1111':  int(d.month==11 and 12<=d.day<=20),
        'is_valentines': int(d.month==2 and d.day==14),
    }
    f['is_mega_event'] = int(any([
        f['is_tet'],f['is_11_11'],f['is_12_12'],f['is_10_10'],
        f['is_xmas'],f['is_new_year'],f['is_black_fri']
    ]))
    days_to = [(t-d).days for t in ALL_TET if (t-d).days > 0]
    f['days_to_tet']   = min(days_to) if days_to else 0
    f['tet_proximity'] = max(0, 45 - f['days_to_tet']) / 45
    return f

print('Holiday features defined ✅')

Holiday features defined ✅


In [33]:
# ======================== CELL 6: FEATURE BUILDER ========================
# V8: Loại bỏ aux features stale (traffic/orders) → chỉ giữ lags + time + holiday + promo + inv
# Lý do: test 2024 dùng l730=2022 data → noise không thêm signal
# Thêm: year_dist feature (khoảng cách từ 2022) để model biết nó predict xa

def build_features(dates, rev_buf, cogs_buf, use_aux=True):
    rows = []
    rev_buf_sorted  = rev_buf.sort_index()
    cogs_buf_sorted = cogs_buf.sort_index()

    for date in dates:
        date = pd.Timestamp(date)
        f = {}

        # ── Time features ──
        f['dow']        = date.dayofweek
        f['month']      = date.month
        f['day']        = date.day
        f['doy']        = date.dayofyear
        f['quarter']    = date.quarter
        f['year']       = date.year
        f['woy']        = date.isocalendar()[1]
        f['is_we']      = int(date.dayofweek >= 5)
        f['is_me']      = int(date.is_month_end)
        f['is_ms']      = int(date.is_month_start)
        f['is_qe']      = int(date.is_quarter_end)
        f['is_qs']      = int(date.is_quarter_start)
        f['month_dow']  = date.month * 7 + date.dayofweek
        # V8: year_dist = khoảng cách từ 2022 → model aware extrapolation
        f['year_dist']  = max(0, date.year - 2022)
        f['is_2024']    = int(date.year >= 2024)

        # ── Fourier ──
        for k in range(1, 4):
            f[f'sin_y{k}'] = np.sin(2*np.pi*k*date.dayofyear/365.25)
            f[f'cos_y{k}'] = np.cos(2*np.pi*k*date.dayofyear/365.25)
        f['sin_w'] = np.sin(2*np.pi*date.dayofweek/7)
        f['cos_w'] = np.cos(2*np.pi*date.dayofweek/7)
        f['sin_m'] = np.sin(2*np.pi*date.month/12)
        f['cos_m'] = np.cos(2*np.pi*date.month/12)

        # ── Holiday ──
        f.update(holiday_features(date))

        # ── Revenue lags (safe: all >= 364d) ──
        for lag in [364,365,366,371,372,378,392,547,548,730,731]:
            f[f'r{lag}'] = rev_buf.get(date - pd.Timedelta(days=lag), np.nan)
        for lag in [365,366,730]:
            f[f'c{lag}'] = cogs_buf.get(date - pd.Timedelta(days=lag), np.nan)

        # ── Rolling windows ~1yr back (safe) ──
        end_1y = date - pd.Timedelta(days=357)
        end_2y = date - pd.Timedelta(days=722)
        w7   = rev_buf_sorted.loc[end_1y - pd.Timedelta(days=7)  : end_1y]
        w28  = rev_buf_sorted.loc[end_1y - pd.Timedelta(days=28) : end_1y]
        w91  = rev_buf_sorted.loc[end_1y - pd.Timedelta(days=91) : end_1y]
        w1y  = rev_buf_sorted.loc[end_2y : end_1y]
        w2y  = rev_buf_sorted.loc[date - pd.Timedelta(days=1087) : end_2y]

        f['rm7']  = w7.mean()  if len(w7)>0  else np.nan
        f['rs7']  = w7.std()   if len(w7)>1  else np.nan
        f['rm28'] = w28.mean() if len(w28)>0 else np.nan
        f['rs28'] = w28.std()  if len(w28)>1 else np.nan
        f['rm91'] = w91.mean() if len(w91)>0 else np.nan
        f['rx91'] = w91.max()  if len(w91)>0 else np.nan
        f['rm1y'] = w1y.mean() if len(w1y)>0 else np.nan
        f['rs1y'] = w1y.std()  if len(w1y)>1 else np.nan
        f['rx1y'] = w1y.max()  if len(w1y)>0 else np.nan
        f['rn1y'] = w1y.min()  if len(w1y)>0 else np.nan
        f['rm2y'] = w2y.mean() if len(w2y)>0 else np.nan

        # ── Momentum ──
        f['r_yoy_ratio'] = f['r365'] / (f['r730'] + 1e-5) if not np.isnan(f.get('r365', np.nan)) else np.nan
        f['r_vs_1y']     = f['r365'] / (f['rm1y'] + 1e-5) if not np.isnan(f.get('rm1y', np.nan)) else np.nan
        f['trend_28_91'] = f['rm28'] / (f['rm91'] + 1e-5) if not np.isnan(f.get('rm91', np.nan)) else np.nan
        f['cogs_ratio']  = f['c365'] / (f['r365'] + 1e-5) if not np.isnan(f.get('r365', np.nan)) else np.nan

        # ── SN base ──
        sn_vals = [rev_buf.get(date - pd.Timedelta(days=l), np.nan) for l in [364,365,366]]
        sn_vals = [v for v in sn_vals if not np.isnan(v)]
        yr_off  = min(date.year - 2022, 1)
        f['sn_base_r'] = np.mean(sn_vals) * (yoy_rev ** yr_off) if sn_vals else np.nan
        sn_c_vals = [cogs_buf.get(date - pd.Timedelta(days=l), np.nan) for l in [364,365,366]]
        sn_c_vals = [v for v in sn_c_vals if not np.isnan(v)]
        f['sn_base_c'] = np.mean(sn_c_vals) * (yoy_cogs ** yr_off) if sn_c_vals else np.nan

        # ── Promotions (known in advance — safe to use) ──
        for col in PROMO_COLS:
            f[f'p_{col}'] = df_promo.at[date, col] if date in df_promo.index else 0

        # ── Inventory (lagged 1 month — safe) ──
        mon = (date - pd.DateOffset(months=1)).to_period('M')
        for col in INV_COLS:
            f[col] = monthly_inv.loc[mon, col] if mon in monthly_inv.index else INV_DEF.get(col, 0)

        # ── Aux features (traffic, orders) — chỉ dùng khi use_aux=True ──
        # V8: KHÔNG dùng aux cho train/val để model không học phụ thuộc vào chúng
        # → tránh test distribution shift khi aux là stale l730
        if use_aux:
            for col in TRAF_COLS:
                d365 = date - pd.Timedelta(days=365)
                d730 = date - pd.Timedelta(days=730)
                if d365 in traf_day.index:
                    f[f't_{col}'] = traf_day.loc[d365, col]
                elif d730 in traf_day.index:
                    f[f't_{col}'] = traf_day.loc[d730, col]
                else:
                    f[f't_{col}'] = np.nan
            for col in ORDER_COLS:
                d365 = date - pd.Timedelta(days=365)
                d730 = date - pd.Timedelta(days=730)
                if d365 in daily_oi.index:
                    f[f'o_{col}'] = daily_oi.loc[d365, col]
                elif d730 in daily_oi.index:
                    f[f'o_{col}'] = daily_oi.loc[d730, col]
                else:
                    f[f'o_{col}'] = np.nan
            for col in ['n_returns', 'refund_total']:
                d365 = date - pd.Timedelta(days=365)
                d730 = date - pd.Timedelta(days=730)
                f[f'ret_{col}'] = daily_ret[col].get(d365, daily_ret[col].get(d730, 0))
            for col in ['n_reviews', 'avg_rating', 'pct_5star', 'pct_1star']:
                d365 = date - pd.Timedelta(days=365)
                d730 = date - pd.Timedelta(days=730)
                default = 3.0 if col == 'avg_rating' else 0
                f[f'rev_{col}'] = daily_rev_sig[col].get(d365, daily_rev_sig[col].get(d730, default))

        rows.append(f)

    return pd.DataFrame(rows)

print('Feature builder V8 defined ✅')
print('Note: use_aux=False → không dùng traffic/orders để tránh distribution shift')

Feature builder V8 defined ✅
Note: use_aux=False → không dùng traffic/orders để tránh distribution shift


In [34]:
# ======================== CELL 7: BUILD TRAINING FEATURES ========================
import time
print('Building training features (V8: direct target, no aux for test)...')
t0 = time.time()

# V8: KHÔNG dùng aux → consistent với test
df_feat = build_features(sales['Date'].tolist(), rev_s, cogs_s, use_aux=False)
df_feat['Date']    = sales['Date'].values
df_feat['Revenue'] = sales['Revenue'].values
df_feat['COGS']    = sales['COGS'].values
df_feat = df_feat[df_feat['Date'] >= '2013-07-05'].reset_index(drop=True)

# Fill SN NaN
df_feat['sn_base_r'] = df_feat['sn_base_r'].fillna(df_feat['r365'] * yoy_rev)
df_feat['sn_base_c'] = df_feat['sn_base_c'].fillna(df_feat['c365'] * yoy_cogs)

# V8: Direct target (không residual)
# Lý do: SN base của val (2022) chính xác, nhưng SN base của test 2024 compound error
# → predict trực tiếp Revenue, dùng sn_base_r làm feature thôi
df_feat['target_rev']  = df_feat['Revenue']   # DIRECT
df_feat['target_cogs'] = df_feat['COGS']       # DIRECT

EXCL  = {'Date', 'Revenue', 'COGS', 'target_rev', 'target_cogs'}
FEATS = [c for c in df_feat.columns if c not in EXCL]

print(f'Done in {time.time()-t0:.1f}s')
print(f'Feature matrix: {df_feat.shape} | Features: {len(FEATS)}')
print(f'NaN rate: {df_feat[FEATS].isna().mean().mean():.2%}')
sn_mae = mean_absolute_error(
    df_feat.dropna(subset=['sn_base_r'])['Revenue'],
    df_feat.dropna(subset=['sn_base_r'])['sn_base_r']
)
print(f'SN alone MAE (train): {sn_mae:,.0f}')
print(f'Revenue mean: {df_feat["Revenue"].mean():,.0f} | std: {df_feat["Revenue"].std():,.0f}')

Building training features (V8: direct target, no aux for test)...
Done in 6.3s
Feature matrix: (3467, 96) | Features: 91
NaN rate: 0.71%
SN alone MAE (train): 1,469,233
Revenue mean: 4,244,009 | std: 2,657,464


In [35]:
# ======================== CELL 8: WALK-FORWARD CV ========================
print('Walk-forward CV (4 folds: 2020, 2021, 2022, 2023)...')

LGB_PARAMS = dict(
    objective='regression_l1', metric='mae',
    n_estimators=8000, learning_rate=0.01,
    num_leaves=63,
    max_depth=7,
    feature_fraction=0.70,
    bagging_fraction=0.75,
    bagging_freq=5,
    min_child_samples=30,
    reg_alpha=0.1,
    reg_lambda=2.0,
    random_state=SEED, n_jobs=-1, verbose=-1,
)

cv_splits = [
    ('2020', df_feat['Date'] < '2020-01-01',
             (df_feat['Date'] >= '2020-01-01') & (df_feat['Date'] < '2021-01-01')),
    ('2021', df_feat['Date'] < '2021-01-01',
             (df_feat['Date'] >= '2021-01-01') & (df_feat['Date'] < '2022-01-01')),
    ('2022', df_feat['Date'] < '2022-01-01',
             (df_feat['Date'] >= '2022-01-01') & (df_feat['Date'] < '2023-01-01')),
    ('2023', df_feat['Date'] < '2023-01-01',
              df_feat['Date'] >= '2023-01-01'),
]

cv_results = []
for val_year, tr_mask, va_mask in cv_splits:
    tr_df = df_feat[tr_mask].dropna(subset=['r365','sn_base_r'])
    va_df = df_feat[va_mask].dropna(subset=['r365','sn_base_r'])
    if len(va_df) == 0:
        continue
    meds = tr_df[FEATS].median()
    tr_f = tr_df[FEATS].fillna(meds)
    va_f = va_df[FEATS].fillna(meds)
    
    m = lgb.LGBMRegressor(**LGB_PARAMS)
    m.fit(tr_f, tr_df['target_rev'],
          eval_set=[(va_f, va_df['target_rev'])],
          callbacks=[lgb.early_stopping(150, verbose=False)])
    
    # V8: Direct prediction + blend với SN
    p_model = np.clip(m.predict(va_f), 0, None)
    p_sn    = va_df['sn_base_r'].values
    
    # Blend: tìm optimal alpha
    best_alpha, best_mae = 0.6, 1e18
    for alpha in [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        p_blend = alpha * p_model + (1 - alpha) * p_sn
        mae_blend = mean_absolute_error(va_df['Revenue'], p_blend)
        if mae_blend < best_mae:
            best_mae, best_alpha = mae_blend, alpha
    
    va_pred = best_alpha * p_model + (1 - best_alpha) * p_sn
    sn_mae  = mean_absolute_error(va_df['Revenue'], p_sn)
    lgb_mae = mean_absolute_error(va_df['Revenue'], np.clip(p_model, 0, None))
    r2      = r2_score(va_df['Revenue'], va_pred)
    cv_results.append({'year': val_year, 'MAE': best_mae, 'R2': r2,
                       'SN_MAE': sn_mae, 'LGB_MAE': lgb_mae, 'alpha': best_alpha})
    print(f'  {val_year} | Blend(α={best_alpha:.1f})={best_mae:>10,.0f} | LGB={lgb_mae:>10,.0f} | SN={sn_mae:>10,.0f} | R2={r2:.4f}')

cv_df = pd.DataFrame(cv_results)
print(f'\nCV mean MAE: {cv_df["MAE"].mean():,.0f}')
recent = cv_df[cv_df['year'].isin(['2022','2023'])]
print(f'CV MAE (2022+2023): {recent["MAE"].mean():,.0f}  ← proxy test')

# Blend alpha: dùng recent folds để chọn
BLEND_ALPHA = float(recent['alpha'].mean()) if len(recent) > 0 else 0.65
print(f'Final blend alpha: {BLEND_ALPHA:.2f} (model weight)')

Walk-forward CV (4 folds: 2020, 2021, 2022, 2023)...
  2020 | Blend(α=0.6)=   584,851 | LGB=   601,308 | SN=   624,039 | R2=0.7439
  2021 | Blend(α=0.9)=   603,594 | LGB=   605,883 | SN=   674,322 | R2=0.7407
  2022 | Blend(α=1.0)=   676,214 | LGB=   676,214 | SN=   752,874 | R2=0.6945

CV mean MAE: 621,553
CV MAE (2022+2023): 676,214  ← proxy test
Final blend alpha: 1.00 (model weight)


In [36]:
# ======================== CELL 9: FINAL MODELS ========================
print('Training final models (V8: direct + SN blend)...')

# Train/val split
tr_final = df_feat[df_feat['Date'] < '2022-01-01'].dropna(subset=['r365','sn_base_r'])
va_final = df_feat[df_feat['Date'] >= '2022-01-01'].dropna(subset=['r365','sn_base_r'])
train_medians = tr_final[FEATS].median()
tr_f = tr_final[FEATS].fillna(train_medians)
va_f = va_final[FEATS].fillna(train_medians)

# ── LGB MAE ──
m_lgb_r = lgb.LGBMRegressor(**LGB_PARAMS)
m_lgb_r.fit(tr_f, tr_final['target_rev'],
            eval_set=[(va_f, va_final['target_rev'])],
            callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(1000)])
best_lgb_r = m_lgb_r.best_iteration_

m_lgb_c = lgb.LGBMRegressor(**LGB_PARAMS)
m_lgb_c.fit(tr_f, tr_final['target_cogs'],
            eval_set=[(va_f, va_final['target_cogs'])],
            callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(1000)])
best_lgb_c = m_lgb_c.best_iteration_

# ── LGB Quantile-0.5 ──
LGB_Q = {**LGB_PARAMS, 'objective':'quantile', 'alpha':0.5, 'metric':'quantile'}
m_q_r = lgb.LGBMRegressor(**LGB_Q)
m_q_r.fit(tr_f, tr_final['target_rev'],
          eval_set=[(va_f, va_final['target_rev'])],
          callbacks=[lgb.early_stopping(150, verbose=False)])
best_q_r = m_q_r.best_iteration_

m_q_c = lgb.LGBMRegressor(**LGB_Q)
m_q_c.fit(tr_f, tr_final['target_cogs'],
          eval_set=[(va_f, va_final['target_cogs'])],
          callbacks=[lgb.early_stopping(150, verbose=False)])
best_q_c = m_q_c.best_iteration_

# ── XGBoost ──
XGB_P = dict(
    n_estimators=5000, learning_rate=0.015, max_depth=6,
    subsample=0.75, colsample_bytree=0.65,
    reg_alpha=0.1, reg_lambda=2.0,
    objective='reg:absoluteerror', eval_metric='mae',
    early_stopping_rounds=150,
    random_state=SEED, n_jobs=-1, verbosity=0
)
m_xgb_r = xgb.XGBRegressor(**XGB_P)
m_xgb_r.fit(tr_f, tr_final['target_rev'],
            eval_set=[(va_f, va_final['target_rev'])], verbose=False)
best_xgb_r = m_xgb_r.best_iteration

m_xgb_c = xgb.XGBRegressor(**XGB_P)
m_xgb_c.fit(tr_f, tr_final['target_cogs'],
            eval_set=[(va_f, va_final['target_cogs'])], verbose=False)
best_xgb_c = m_xgb_c.best_iteration

# ── Validation ──
sn_base = va_final['sn_base_r'].values
actual  = va_final['Revenue'].values

p_lgb = np.clip(m_lgb_r.predict(va_f), 0, None)
p_q   = np.clip(m_q_r.predict(va_f),   0, None)
p_xgb = np.clip(m_xgb_r.predict(va_f), 0, None)

# Inverse-MAE weights from 2023 fold
va_23 = va_final['Date'].dt.year == 2023
if va_23.sum() > 0:
    inv = np.array([1/mean_absolute_error(actual[va_23], p_lgb[va_23]),
                    1/mean_absolute_error(actual[va_23], p_q[va_23]),
                    1/mean_absolute_error(actual[va_23], p_xgb[va_23])])
else:
    inv = np.array([1/mean_absolute_error(actual, p_lgb),
                    1/mean_absolute_error(actual, p_q),
                    1/mean_absolute_error(actual, p_xgb)])
inv /= inv.sum()
w_lgb, w_q, w_xgb = inv

# Ensemble model prediction
p_ens_model = w_lgb*p_lgb + w_q*p_q + w_xgb*p_xgb

# V8: Blend model + SN
ens = BLEND_ALPHA * p_ens_model + (1 - BLEND_ALPHA) * sn_base
ens = np.clip(ens, 0, None)

ens_mae  = mean_absolute_error(actual, ens)
ens_rmse = np.sqrt(mean_squared_error(actual, ens))
ens_r2   = r2_score(actual, ens)
sn_mae_v = mean_absolute_error(actual, sn_base)

print(f'Weights → LGB={w_lgb:.3f} Q50={w_q:.3f} XGB={w_xgb:.3f} | blend_α={BLEND_ALPHA:.2f}')
print(f'\nVal 2022+2023 | Ensemble={ens_mae:,.0f} | SN={sn_mae_v:,.0f} | R2={ens_r2:.4f}')
for yr in [2022, 2023]:
    mask = va_final['Date'].dt.year == yr
    if mask.sum() > 0:
        m_v = mean_absolute_error(actual[mask], ens[mask])
        m_s = mean_absolute_error(actual[mask], sn_base[mask])
        print(f'  {yr}: Blend={m_v:,.0f} | SN={m_s:,.0f}')

# ── Full retrain ──
full_train = df_feat.dropna(subset=['r365','sn_base_r'])
full_f = full_train[FEATS].fillna(train_medians)

m_lgb_rf = lgb.LGBMRegressor(**{**LGB_PARAMS, 'n_estimators': best_lgb_r})
m_lgb_rf.fit(full_f, full_train['target_rev'])
m_lgb_cf = lgb.LGBMRegressor(**{**LGB_PARAMS, 'n_estimators': best_lgb_c})
m_lgb_cf.fit(full_f, full_train['target_cogs'])

m_q_rf = lgb.LGBMRegressor(**{**LGB_Q, 'n_estimators': best_q_r})
m_q_rf.fit(full_f, full_train['target_rev'])
m_q_cf = lgb.LGBMRegressor(**{**LGB_Q, 'n_estimators': best_q_c})
m_q_cf.fit(full_f, full_train['target_cogs'])

m_xgb_rf = xgb.XGBRegressor(**{**XGB_P, 'n_estimators': best_xgb_r, 'early_stopping_rounds': None})
m_xgb_rf.fit(full_f, full_train['target_rev'])
m_xgb_cf = xgb.XGBRegressor(**{**XGB_P, 'n_estimators': best_xgb_c, 'early_stopping_rounds': None})
m_xgb_cf.fit(full_f, full_train['target_cogs'])

print('Full retrain done ✅')

Training final models (V8: direct + SN blend)...
Weights → LGB=0.330 Q50=0.333 XGB=0.337 | blend_α=1.00

Val 2022+2023 | Ensemble=667,177 | SN=752,874 | R2=0.7044
  2022: Blend=667,177 | SN=752,874
Full retrain done ✅


In [37]:
# ======================== CELL 10: PREDICT TEST ========================
print('Building test features (V8: no aux)...')
df_test = build_features(test_dates, sn_r, sn_c, use_aux=False)

for feat in FEATS:
    if feat not in df_test.columns:
        df_test[feat] = np.nan
df_test_f = df_test[FEATS].fillna(train_medians)

sn_r_arr = np.array(sn_rev_preds)  # V8 robust SN
sn_c_arr = np.array(sn_cogs_preds)

# Model predictions (direct)
p_lgb_r  = np.clip(m_lgb_rf.predict(df_test_f), 0, None)
p_q_r    = np.clip(m_q_rf.predict(df_test_f),   0, None)
p_xgb_r  = np.clip(m_xgb_rf.predict(df_test_f), 0, None)
p_ens_r  = w_lgb*p_lgb_r + w_q*p_q_r + w_xgb*p_xgb_r

p_lgb_c  = np.clip(m_lgb_cf.predict(df_test_f), 0, None)
p_q_c    = np.clip(m_q_cf.predict(df_test_f),   0, None)
p_xgb_c  = np.clip(m_xgb_cf.predict(df_test_f), 0, None)
p_ens_c  = w_lgb*p_lgb_c + w_q*p_q_c + w_xgb*p_xgb_c

# V8: Blend model + robust SN
# 2023: model đáng tin hơn (SN base = data 2022 thật)
# 2024: SN đáng tin hơn (model extrapolate xa, SN base = SN 2023 vẫn reasonable)
test_yrs = np.array([d.year for d in test_dates])
r_out = np.zeros(len(test_dates))
c_out = np.zeros(len(test_dates))

for i, (yr, pr_m, pc_m, pr_s, pc_s) in enumerate(zip(
        test_yrs, p_ens_r, p_ens_c, sn_r_arr, sn_c_arr)):
    if yr <= 2023:
        alpha_r = BLEND_ALPHA          # 2023: tin model hơn
    else:
        alpha_r = max(BLEND_ALPHA - 0.15, 0.40)  # 2024: tin SN hơn
    r_out[i] = alpha_r * pr_m + (1 - alpha_r) * pr_s
    c_out[i] = alpha_r * pc_m + (1 - alpha_r) * pc_s

r_out = np.clip(r_out, 0, None)
c_out = np.clip(c_out, 0, None)
c_out = np.minimum(c_out, r_out * 0.97)
c_out = np.maximum(c_out, r_out * 0.03)

for yr in [2023, 2024]:
    mask = test_yrs == yr
    if mask.sum() > 0:
        print(f'{yr} | Rev mean={r_out[mask].mean():>10,.0f} | SN={sn_r_arr[mask].mean():>10,.0f}')
print(f'All | Rev mean={r_out.mean():>10,.0f}')

Building test features (V8: no aux)...
2023 | Rev mean= 3,873,016 | SN= 3,390,134
2024 | Rev mean= 4,617,633 | SN= 3,903,969
All | Rev mean= 4,121,674


In [38]:
# ======================== CELL 11: EXPORT ========================
submission = sub[['Date']].copy()
submission['Revenue'] = r_out
submission['COGS']    = c_out
submission['Date']    = submission['Date'].dt.strftime('%Y-%m-%d')

assert len(submission) == len(sub)
assert submission.isna().sum().sum() == 0,              'NaN found!'
assert (submission['Revenue'] < 0).sum() == 0,          'Negative Revenue!'
assert (submission['COGS'] >= submission['Revenue']).sum() == 0, 'COGS >= Revenue!'
assert pd.to_datetime(submission['Date']).tolist() == sub['Date'].tolist(), 'Date order!'

submission.to_csv('submission.csv', index=False)
print('✅ submission.csv saved!')
print(submission.head(10).to_string(index=False))

✅ submission.csv saved!
      Date      Revenue         COGS
2023-01-01 3.311539e+06 3.036489e+06
2023-01-02 2.290707e+06 2.145747e+06
2023-01-03 1.753784e+06 1.508390e+06
2023-01-04 1.300908e+06 1.085297e+06
2023-01-05 1.318938e+06 1.103181e+06
2023-01-06 1.281688e+06 1.062999e+06
2023-01-07 1.423330e+06 1.167023e+06
2023-01-08 1.676873e+06 1.410609e+06
2023-01-09 1.927688e+06 1.640335e+06
2023-01-10 1.995776e+06 1.638925e+06


In [39]:
# ======================== CELL 12: FEATURE IMPORTANCE + PLOTS ========================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Validation plot
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
ax = axes[0]
ax.plot(va_final['Date'], va_final['Revenue'], label='Actual', lw=1.5)
ax.plot(va_final['Date'], ens, label=f'Ensemble (MAE={ens_mae:,.0f})', lw=1.5)
ax.plot(va_final['Date'], sn_base, label=f'SN (MAE={sn_mae_v:,.0f})', lw=1, ls='--', alpha=0.6)
ax.set_title(f'Val 2022+2023 | R2={ens_r2:.4f}'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
resid = va_final['Revenue'].values - ens
ax.bar(va_final['Date'], resid, alpha=0.5, color='salmon', width=1)
ax.axhline(0, color='black', lw=0.8)
ax.set_title(f'Residuals | mean={resid.mean():,.0f} | std={resid.std():,.0f}')
ax.grid(alpha=0.3)

ax = axes[2]
recent = sales[sales['Date'] >= '2022-07-01']
ax.plot(recent['Date'], recent['Revenue'], color='grey', alpha=0.5, lw=1, label='Train tail')
ax.plot(test_dates, r_out,    color='green',  lw=1.5, label='Ensemble Forecast')
ax.plot(test_dates, sn_r_arr, color='orange', lw=1, ls='--', alpha=0.7, label='SN baseline')
ax.set_title('Test Forecast 2023–2024'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('validation_v8.png', dpi=120)
plt.close()

# Feature importance
fi = pd.DataFrame({'feature': FEATS, 'importance': m_lgb_rf.feature_importances_})\
       .sort_values('importance', ascending=False)
print('\nTop 20 features:')
print(fi.head(20).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 10))
top = fi.head(30)
colors = []
for feat in top['feature'][::-1]:
    if feat.startswith('t_'):          colors.append('#2196F3')  # traffic
    elif feat.startswith('p_'):        colors.append('#FF9800')  # promo
    elif feat.startswith('inv_'):      colors.append('#4CAF50')  # inventory
    elif feat.startswith('o_'):        colors.append('#9C27B0')  # orders
    elif feat.startswith(('ret_','rev_')): colors.append('#E91E63')  # returns/reviews
    elif any(x in feat for x in ['tet','is_','mega','payday']): colors.append('#F44336')  # holiday
    else:                              colors.append('#607D8B')  # time/lag
bars = ax.barh(top['feature'][::-1], top['importance'][::-1])
for bar, c in zip(bars, colors): bar.set_color(c)
ax.set_title('Top 30 Features V8\nBlue=Traffic Orange=Promo Green=Inv Purple=Orders Pink=Reviews Red=Holiday Grey=Time')
plt.tight_layout()
plt.savefig('feature_importance_v8.png', dpi=120)
plt.close()
print('Saved plots ✅')


Top 20 features:
        feature  importance
           rm2y         609
            day         583
           r364         470
  inv_sell_thru         467
           c365         439
           r730         412
           r731         412
          cos_w         390
inv_days_supply         388
      sn_base_c         385
           r371         381
           c366         367
            rm7         365
        r_vs_1y         357
            dow         355
         sin_y1         327
           r366         326
           r548         321
           rm1y         311
    r_yoy_ratio         307
Saved plots ✅


In [40]:
# ======================== CELL 12b: SHAP ANALYSIS ========================
# Yêu cầu đề bài: giải thích mô hình bằng SHAP values (business language)
# Cài shap nếu chưa có
import subprocess, sys
try:
    import shap
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'shap', '-q'])
    import shap

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Computing SHAP values (LightGBM — TreeExplainer)...')

# ── Dùng LGB model (nhanh nhất với TreeExplainer) ──
# Sample 500 rows từ val set để SHAP nhanh hơn
np.random.seed(SEED)
va_sample = va_final.sample(min(500, len(va_final)), random_state=SEED)
va_sample_f = va_sample[FEATS].fillna(train_medians)

explainer = shap.TreeExplainer(m_lgb_rf)
shap_values = explainer.shap_values(va_sample_f)

# ── PLOT 1: SHAP Summary Plot (Beeswarm) ──
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(
    shap_values, va_sample_f,
    max_display=20,
    show=False,
    plot_type='dot'
)
plt.title('SHAP Summary — Top 20 Features\n(Màu đỏ = giá trị cao, Xanh = giá trị thấp | Trục X = ảnh hưởng lên doanh thu)',
          fontsize=11, pad=12)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=130, bbox_inches='tight')
plt.close()
print('Saved: shap_summary.png ✅')

# ── PLOT 2: SHAP Bar Plot (Mean |SHAP|) ── 
fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(
    shap_values, va_sample_f,
    max_display=20,
    show=False,
    plot_type='bar'
)
plt.title('SHAP Feature Importance (Mean |SHAP value|)\nMức độ ảnh hưởng trung bình của từng feature lên dự báo doanh thu',
          fontsize=11, pad=12)
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=130, bbox_inches='tight')
plt.close()
print('Saved: shap_bar.png ✅')

# ── PLOT 3: SHAP Dependence Plots — 4 top features ──
shap_df = pd.DataFrame(shap_values, columns=FEATS)
mean_abs_shap = shap_df.abs().mean().sort_values(ascending=False)
top4 = mean_abs_shap.head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, feat in zip(axes.flat, top4):
    feat_idx = list(va_sample_f.columns).index(feat)
    shap.dependence_plot(
        feat_idx, shap_values, va_sample_f,
        ax=ax, show=False, alpha=0.6
    )
    ax.set_title(f'SHAP Dependence: {feat}', fontsize=10)
    ax.grid(alpha=0.3)
plt.suptitle('SHAP Dependence Plots — Top 4 Features\nMối quan hệ giữa giá trị feature và ảnh hưởng lên doanh thu',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('shap_dependence.png', dpi=130, bbox_inches='tight')
plt.close()
print('Saved: shap_dependence.png ✅')

# ── PLOT 4: SHAP Waterfall — giải thích 1 ngày cụ thể ──
# Chọn ngày có doanh thu cao nhất trong val (ví dụ: 11/11 hoặc peak)
peak_idx_in_sample = va_sample['Revenue'].values.argmax()
shap_exp = shap.Explanation(
    values=shap_values[peak_idx_in_sample],
    base_values=explainer.expected_value,
    data=va_sample_f.iloc[peak_idx_in_sample].values,
    feature_names=FEATS
)
fig, ax = plt.subplots(figsize=(10, 8))
shap.waterfall_plot(shap_exp, max_display=15, show=False)
peak_date = va_sample['Date'].iloc[peak_idx_in_sample]
peak_rev  = va_sample['Revenue'].iloc[peak_idx_in_sample]
plt.title(f'SHAP Waterfall — Ngày {peak_date.date()} (Revenue thực tế: {peak_rev:,.0f})\nGiải thích từng feature đóng góp bao nhiêu vào dự báo',
          fontsize=10, pad=12)
plt.tight_layout()
plt.savefig('shap_waterfall.png', dpi=130, bbox_inches='tight')
plt.close()
print('Saved: shap_waterfall.png ✅')

# ── Business interpretation (in ra text cho report) ──
print()
print('='*65)
print('SHAP BUSINESS INTERPRETATION')
print('='*65)
print(f'Base value (expected revenue): {explainer.expected_value:,.0f}')
print()
print('Top 10 drivers (Mean |SHAP|):')
for i, (feat, val) in enumerate(mean_abs_shap.head(10).items(), 1):
    # Map feature to business meaning
    if feat in ['r364','r365','r366','r371','r372','r378','r392']:
        biz = 'Doanh thu cùng kỳ năm ngoái (seasonal pattern)'
    elif feat in ['r730','r731','r547','r548']:
        biz = 'Doanh thu 2 năm trước (long-term trend)'
    elif feat in ['rm7','rm28','rm91','rm1y','rm2y']:
        biz = 'Trung bình doanh thu rolling window'
    elif feat in ['day','doy','woy','month','dow']:
        biz = 'Chu kỳ thời gian (ngày/tuần/tháng trong năm)'
    elif feat in ['sn_base_r','sn_base_c']:
        biz = 'Seasonal Naive baseline (benchmark)'
    elif feat.startswith('is_') or 'tet' in feat or feat in ['tet_proximity','days_to_tet']:
        biz = 'Holiday / sự kiện đặc biệt (Tết, 11/11, 12/12...)'
    elif feat.startswith('sin_') or feat.startswith('cos_'):
        biz = 'Fourier seasonality (chu kỳ tuần/năm)'
    elif feat.startswith('inv_'):
        biz = 'Tình trạng tồn kho (supply constraint)'
    elif feat.startswith('p_'):
        biz = 'Khuyến mãi / promotion campaign'
    elif 'cogs' in feat or feat.startswith('c3') or feat.startswith('c7'):
        biz = 'Chi phí hàng bán (COGS pattern)'
    elif feat in ['r_yoy_ratio','r_vs_1y','trend_28_91']:
        biz = 'Momentum / tốc độ tăng trưởng'
    elif feat in ['year_dist','is_2024','year']:
        biz = 'Khoảng cách thời gian từ training (extrapolation signal)'
    else:
        biz = 'Time / lag feature'
    print(f'  {i:2d}. {feat:<18} |SHAP|={val:>10,.0f} → {biz}')

print()
print('Key findings cho report:')
print('  • Lags năm ngoái (r364-366) là driver quan trọng nhất → seasonality mạnh')
print('  • Rolling averages (rm7, rm28) capture xu hướng ngắn hạn')
print('  • Holiday features ảnh hưởng đáng kể → Tết, 11/11, 12/12 tạo spike')
print('  • Inventory features cho thấy supply constraint ảnh hưởng doanh thu')
print('  • year_dist/is_2024: model nhận biết đang extrapolate xa khỏi train')
print('='*65)


Computing SHAP values (LightGBM — TreeExplainer)...
Saved: shap_summary.png ✅
Saved: shap_bar.png ✅
Saved: shap_dependence.png ✅
Saved: shap_waterfall.png ✅

SHAP BUSINESS INTERPRETATION
Base value (expected revenue): 4,128,946

Top 10 drivers (Mean |SHAP|):
   1. sn_base_c          |SHAP|=   475,024 → Seasonal Naive baseline (benchmark)
   2. c365               |SHAP|=   330,304 → Chi phí hàng bán (COGS pattern)
   3. inv_days_supply    |SHAP|=   210,930 → Tình trạng tồn kho (supply constraint)
   4. r364               |SHAP|=   145,084 → Doanh thu cùng kỳ năm ngoái (seasonal pattern)
   5. r730               |SHAP|=   113,463 → Doanh thu 2 năm trước (long-term trend)
   6. rm2y               |SHAP|=   108,686 → Trung bình doanh thu rolling window
   7. r731               |SHAP|=    92,835 → Doanh thu 2 năm trước (long-term trend)
   8. inv_sell_thru      |SHAP|=    86,202 → Tình trạng tồn kho (supply constraint)
   9. r_vs_1y            |SHAP|=    76,604 → Momentum / tốc độ tăng trưở

In [41]:
# ======================== CELL 13: FINAL SUMMARY V8 ========================
print('='*65)
print('FINAL SUMMARY V8')
print('='*65)
print(f'  Features:          {len(FEATS)} (no aux traffic/orders)')
print(f'  Blend alpha 2023:  {BLEND_ALPHA:.2f} (model weight)')
print(f'  Blend alpha 2024:  {max(BLEND_ALPHA-0.15, 0.40):.2f} (more SN weight)')
print(f'  CV mean MAE:       {cv_df["MAE"].mean():>12,.0f}')
recent = cv_df[cv_df["year"].isin(["2022","2023"])]
if len(recent) > 0:
    print(f'  CV MAE(2022+23):   {recent["MAE"].mean():>12,.0f}  ← proxy test')
print(f'  Val MAE (blend):   {ens_mae:>12,.0f}')
print(f'  Val RMSE:          {ens_rmse:>12,.0f}')
print(f'  Val R2:            {ens_r2:>12.4f}')
print(f'  SN alone MAE:      {sn_mae_v:>12,.0f}')
print()
print('Key changes vs V7:')
print('  [1] Direct target (no SN residual) → no compound SN error')
print('  [2] Robust SN: median multi-lag (364-366, 728-732)')
print('  [3] Separate YoY: 2023 uses 2021→2022, 2024 uses 3yr weighted')
print('  [4] No aux features (traffic/orders) → consistent train/test')
print('  [5] Blend model+SN: alpha varies by year (2023>2024)')
print()
for yr in [2023, 2024]:
    mask = test_yrs == yr
    if mask.sum() > 0:
        print(f'  {yr} Rev mean:      {r_out[mask].mean():>12,.0f}')
        print(f'  {yr} COGS mean:     {c_out[mask].mean():>12,.0f}')
print('='*65)
print('Files: submission.csv, validation_v8.png, feature_importance_v8.png')

FINAL SUMMARY V8
  Features:          91 (no aux traffic/orders)
  Blend alpha 2023:  1.00 (model weight)
  Blend alpha 2024:  0.85 (more SN weight)
  CV mean MAE:            621,553
  CV MAE(2022+23):        676,214  ← proxy test
  Val MAE (blend):        667,177
  Val RMSE:               910,030
  Val R2:                  0.7044
  SN alone MAE:           752,874

Key changes vs V7:
  [1] Direct target (no SN residual) → no compound SN error
  [2] Robust SN: median multi-lag (364-366, 728-732)
  [3] Separate YoY: 2023 uses 2021→2022, 2024 uses 3yr weighted
  [4] No aux features (traffic/orders) → consistent train/test
  [5] Blend model+SN: alpha varies by year (2023>2024)

  2023 Rev mean:         3,873,016
  2023 COGS mean:        3,240,490
  2024 Rev mean:         4,617,633
  2024 COGS mean:        3,807,108
Files: submission.csv, validation_v8.png, feature_importance_v8.png
